# Turkish Legal RAG - Clean Reranker Fine-Tuning

Bu notebook, bizim hazırladığımız temiz reranker datasını kullanır:

- `clean_train.jsonl`
- `clean_dev.jsonl`

Bu versiyonda Kaggle CSV'den tekrar pair üretmiyoruz. Çünkü en son oluşturduğumuz temiz sette pozitifler doğrudan `positive_parent_id` ile bağlı, negatifler ise BM25 hard negative olarak hazırlanmış durumda.

Akış:

1. Kaggle input dosyalarını `/kaggle/working/legal-rag` içine kopyala.
2. `clean_train.jsonl` ve `clean_dev.jsonl` kontrol et.
3. BERTurk cross-encoder reranker fine-tune et.
4. Gold benchmark üzerinde hybrid / pure rerank / fusion skorlarını ölç.


In [ ]:
!nvidia-smi
!echo "Input files:"
!find /kaggle/input -maxdepth 6 -type f | sort | head -300


## 1. Paket Kurulumu


In [ ]:
!pip install -q -U sentence-transformers faiss-cpu datasets tqdm


## 2. Dosyaları Çalışma Klasörüne Kopyala

Bu hücre dosyaları isimlerine göre recursive arar. Kaggle dataset path'i farklı olsa bile çalışması için böyle yaptık.


In [ ]:
from pathlib import Path
import shutil

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/legal-rag")

for path in [
    WORK_DIR / "scripts",
    WORK_DIR / "data/processed",
    WORK_DIR / "data/index",
    WORK_DIR / "data/reranker",
    WORK_DIR / "data/eval",
    WORK_DIR / "models",
]:
    path.mkdir(parents=True, exist_ok=True)

def find_input_file(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"{name} not found under {INPUT_ROOT}. Kaggle dataset'e ekledin mi?")
    return matches[0]

def copy_required(name: str, dest_dir: Path) -> Path:
    src = find_input_file(name)
    dst = dest_dir / name
    shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
    return dst

def copy_optional(name: str, dest_dir: Path) -> Path | None:
    matches = sorted(INPUT_ROOT.rglob(name))
    if not matches:
        print(f"Optional file not found, skipping: {name}")
        return None
    src = matches[0]
    dst = dest_dir / name
    shutil.copy2(src, dst)
    print(f"Copied optional {src} -> {dst}")
    return dst

for name in [
    "train_reranker.py",
    "rerank_search.py",
    "evaluate_retrieval.py",
]:
    copy_required(name, WORK_DIR / "scripts")

for name in ["retrieval_chunks.json", "retrieval_corpus.json"]:
    copy_required(name, WORK_DIR / "data/processed")

for name in ["faiss_bge_m3.index", "metadata_bge_m3.json", "index_config_bge_m3.json"]:
    copy_required(name, WORK_DIR / "data/index")

for name in ["clean_train.jsonl", "clean_dev.jsonl"]:
    copy_required(name, WORK_DIR / "data/reranker")

for name in [
    "clean_reranker_train_questions.csv",
    "clean_reranker_train_questions.stats.json",
    "clean_reranker_pairs.stats.json",
]:
    copy_optional(name, WORK_DIR / "data/reranker")

copy_required("qa_benchmark_gold.csv", WORK_DIR / "data/eval")

print("\nWorking files:")
!find /kaggle/working/legal-rag -maxdepth 4 -type f | sort


## 3. Clean Train / Dev Kontrolü


In [ ]:
import json
from collections import Counter
from pathlib import Path

train_path = WORK_DIR / "data/reranker/clean_train.jsonl"
dev_path = WORK_DIR / "data/reranker/clean_dev.jsonl"

def load_jsonl(path: Path):
    return [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]

train_rows = load_jsonl(train_path)
dev_rows = load_jsonl(dev_path)

def summarize(rows):
    return {
        "rows": len(rows),
        "labels": dict(Counter(row["label"] for row in rows)),
        "query_groups": len({row.get("query_id") for row in rows}),
        "sources": Counter(row.get("question_source_dataset", row.get("source", "")) for row in rows).most_common(10),
    }

print("Train:")
print(json.dumps(summarize(train_rows), ensure_ascii=False, indent=2))
print("\nDev:")
print(json.dumps(summarize(dev_rows), ensure_ascii=False, indent=2))

print("\nPositive sample:")
for row in train_rows:
    if row["label"] == 1:
        print(json.dumps({
            "query": row["query"],
            "candidate_id": row["candidate_id"],
            "parent_id": row.get("parent_id"),
            "citation_label": row["citation_label"],
            "question_source_dataset": row.get("question_source_dataset"),
        }, ensure_ascii=False, indent=2))
        break


## 4. BERTurk Cross-Encoder Reranker Fine-Tuning

Model ayrı klasöre kaydedilir:

`/kaggle/working/legal-rag/models/legal-berturk-reranker-clean`


In [ ]:
!rm -rf /kaggle/working/legal-rag/models/legal-berturk-reranker-clean

!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/legal-rag/scripts/train_reranker.py   --train /kaggle/working/legal-rag/data/reranker/clean_train.jsonl   --dev /kaggle/working/legal-rag/data/reranker/clean_dev.jsonl   --base-model dbmdz/bert-base-turkish-cased   --output-dir /kaggle/working/legal-rag/models/legal-berturk-reranker-clean   --epochs 2   --batch-size 8   --learning-rate 2e-5   --max-length 512   --device cuda   --evaluation-steps 200   --save-best-model


## 5. Model Dosyalarını Kontrol Et


In [ ]:
!ls -lh /kaggle/working/legal-rag/models/legal-berturk-reranker-clean
!test -f /kaggle/working/legal-rag/models/legal-berturk-reranker-clean/model.safetensors || test -f /kaggle/working/legal-rag/models/legal-berturk-reranker-clean/pytorch_model.bin


## 6. Opsiyonel Hızlı Arama Testi

Bu hücre varsayılan olarak kapalı. Önceki notebookta bu demo hücresi model yüklemede bekletebiliyordu. Benchmark için gerekli değil.

Çalıştırmak istersen `RUN_DEMO = True` yap.


In [ ]:
import subprocess

RUN_DEMO = False
RERANKER_MODEL = WORK_DIR / "models/legal-berturk-reranker-clean"

def run_rerank_query(question: str, ranking_mode: str = "rerank"):
    cmd = [
        "python", str(WORK_DIR / "scripts/rerank_search.py"),
        question,
        "--top-k", "5",
        "--index", str(WORK_DIR / "data/index/faiss_bge_m3.index"),
        "--metadata", str(WORK_DIR / "data/index/metadata_bge_m3.json"),
        "--config", str(WORK_DIR / "data/index/index_config_bge_m3.json"),
        "--articles", str(WORK_DIR / "data/processed/retrieval_corpus.json"),
        "--embedding-device", "cpu",
        "--device", "cuda",
        "--reranker-model", str(RERANKER_MODEL),
        "--rerank-batch-size", "8",
        "--ranking-mode", ranking_mode,
        "--show-text",
    ]
    subprocess.run(cmd, check=True)

if RUN_DEMO:
    run_rerank_query("işçi 2 gün işe gelmezse ne olur?", ranking_mode="rerank")
else:
    print("Demo skipped. Benchmark cells are the important part.")


## 7. Gold Benchmark Evaluation

Önce hybrid baseline'ı ölçüyoruz. Sonra yeni clean reranker ile pure rerank ve fusion modlarını ölçüyoruz.


In [ ]:
!python /kaggle/working/legal-rag/scripts/evaluate_retrieval.py   --benchmark /kaggle/working/legal-rag/data/eval/qa_benchmark_gold.csv   --corpus /kaggle/working/legal-rag/data/processed/retrieval_corpus.json   --chunks /kaggle/working/legal-rag/data/processed/retrieval_chunks.json   --index /kaggle/working/legal-rag/data/index/faiss_bge_m3.index   --metadata /kaggle/working/legal-rag/data/index/metadata_bge_m3.json   --config /kaggle/working/legal-rag/data/index/index_config_bge_m3.json   --mode hybrid   --embedding-device cuda   --top-k 10   --output /kaggle/working/legal-rag/data/eval/eval_hybrid_for_clean.json


In [ ]:
!python /kaggle/working/legal-rag/scripts/evaluate_retrieval.py   --benchmark /kaggle/working/legal-rag/data/eval/qa_benchmark_gold.csv   --corpus /kaggle/working/legal-rag/data/processed/retrieval_corpus.json   --chunks /kaggle/working/legal-rag/data/processed/retrieval_chunks.json   --index /kaggle/working/legal-rag/data/index/faiss_bge_m3.index   --metadata /kaggle/working/legal-rag/data/index/metadata_bge_m3.json   --config /kaggle/working/legal-rag/data/index/index_config_bge_m3.json   --mode rerank   --reranker-model /kaggle/working/legal-rag/models/legal-berturk-reranker-clean   --embedding-device cuda   --reranker-device cuda   --rerank-batch-size 8   --top-k 10   --output /kaggle/working/legal-rag/data/eval/eval_rerank_clean.json


In [ ]:
!python /kaggle/working/legal-rag/scripts/evaluate_retrieval.py   --benchmark /kaggle/working/legal-rag/data/eval/qa_benchmark_gold.csv   --corpus /kaggle/working/legal-rag/data/processed/retrieval_corpus.json   --chunks /kaggle/working/legal-rag/data/processed/retrieval_chunks.json   --index /kaggle/working/legal-rag/data/index/faiss_bge_m3.index   --metadata /kaggle/working/legal-rag/data/index/metadata_bge_m3.json   --config /kaggle/working/legal-rag/data/index/index_config_bge_m3.json   --mode rerank_fusion   --reranker-model /kaggle/working/legal-rag/models/legal-berturk-reranker-clean   --embedding-device cuda   --reranker-device cuda   --rerank-batch-size 8   --reranker-weight 0.35   --hybrid-weight 0.65   --top-k 10   --output /kaggle/working/legal-rag/data/eval/eval_rerank_fusion_clean.json


## 8. Eval Sonuçlarını Tek Yerde Göster


In [ ]:
import json

for name in [
    "eval_hybrid_for_clean.json",
    "eval_rerank_clean.json",
    "eval_rerank_fusion_clean.json",
]:
    path = WORK_DIR / "data/eval" / name
    data = json.loads(path.read_text(encoding="utf-8"))
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print(json.dumps(data["summary"], ensure_ascii=False, indent=2))


## 9. Output Olarak Saklanacaklar

Kaggle notebook bitince özellikle şu klasör ve dosyaları output olarak saklayın:

- `/kaggle/working/legal-rag/models/legal-berturk-reranker-clean`
- `/kaggle/working/legal-rag/data/eval/eval_rerank_clean.json`
- `/kaggle/working/legal-rag/data/eval/eval_rerank_fusion_clean.json`

Rapor için karşılaştırma:

```text
dense vs hybrid vs external-augmented reranker vs kaggle-law reranker vs clean reranker
```
